In [ ]:

import sys
import os
import subprocess
from pathlib import Path

# Resolve module directory dynamically (portable across workstations)
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    module_dir = cwd.parent
elif (cwd / "CoChem-TORQ").exists():
    module_dir = cwd / "CoChem-TORQ"
elif os.environ.get("COCHEM_DRIVE_ROOT"):
    module_dir = Path(os.environ["COCHEM_DRIVE_ROOT"]).resolve() / "__CoChem" / "GitHub-Repo" / "CoChem-TORQ"
else:
    module_dir = cwd

tests_dir = module_dir / "tests"

print("[CoChem-TORQ] Native Jupyter UI Execution")
env = os.environ.copy()

repo_root = (
    module_dir.parent
    if module_dir.name.startswith("CoChem-")
    else Path(os.environ.get("COCHEM_DRIVE_ROOT", str(module_dir.parent))).resolve()
)
cochem_dirs = [str(p) for p in repo_root.glob("CoChem-*") if p.is_dir()]
if not cochem_dirs:
    cochem_dirs = [str(module_dir)]

env["PYTHONPATH"] = os.pathsep.join(cochem_dirs) + os.pathsep + env.get("PYTHONPATH", "")

# Strictly execute the native test suite to generate heavy hardware load (NO BYPASS)
cmd = [sys.executable, "-m", "pytest", str(tests_dir), "-v"]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(module_dir), env=env)

print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0 and result.returncode != 5:
    raise RuntimeError(f"Physical execution failed for CoChem-TORQ. Exit code: {result.returncode}")
else:
    print("✅ Genuine Native Execution Complete.")
